# Part 1 — Notebook 02: Read LHE and Plot Generator-Level Kinematics

## Pedagogical Goal & Overview

In **Part 1 — Notebook 02**, you will read raw parton-level LHE event records, reconstruct $Z$ boson kinematics ($p_Z^\mu = p_b^\mu + p_{\bar{b}}^\mu$), compute physical observables ($p_T, \eta, \phi, \Delta R_{b\bar{b}}, m_{b\bar{b}}$), apply Monte Carlo weights, and plot 1D and 2D correlation histograms with Scikit-HEP `pylhe` and `mplhep`.

### Scientific Objectives
1. Parse LHE events non-destructively using `pylhe`.
2. Extract final-state partons ($b, \bar{b}, PID=\pm 5, status=1$) and event weights ($w_i$).
3. Reconstruct four-momentum vectors and compute kinematic observables.
4. Compare $\Delta R_{b\bar{b}}$ against the high-boost analytical guide:
$$\Delta R_{b\bar{b}} \approx \frac{2m_Z}{p_T^Z}$$

## Step 1: Environment Setup & Strict Notebook 01 Output Check

We check whether the 1000-event production LHE file generated in Notebook 01 (`Zbbj_LO/Events/run_01/unweighted_events.lhe.gz`) exists in this Colab runtime.

> [!CAUTION]
> **No Remote Fallback**: If the output from Notebook 01 is missing, you must run `Part1_01_process_to_lhe.ipynb` first in your Google Colab runtime session to generate the LHE file.

In [ ]:
import os
import sys

print("Installing required dependencies (pylhe, mplhep, matplotlib, numpy)...")
!{sys.executable} -m pip install -q pylhe mplhep matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import pylhe

hep.style.use(hep.style.CMS)

# Strict check for Notebook 01 production output
lhe_candidates = [
    "Zbbj_LO/Events/run_01/unweighted_events.lhe.gz",
    "../Zbbj_LO/Events/run_01/unweighted_events.lhe.gz",
    "unweighted_events.lhe.gz"
]

target_lhe = None
for path in lhe_candidates:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        target_lhe = path
        break

if target_lhe is None:
    raise FileNotFoundError(
        "\n\n[ERROR]: Production LHE event file not found!\n"
        "Please execute 'Part1_01_process_to_lhe.ipynb' first in this Google Colab environment "
        "to generate the MadGraph LHE event file before running Notebook 02.\n"
    )

print(f"SUCCESS: Found LHE event file at: {target_lhe}")


> [!IMPORTANT]
> ### Self-Reflection & Coding Checkpoint 2.1
> 1. **Memory Management Question**: Why does `pylhe.read_lhe_with_attributes()` return a Python generator/iterator rather than loading all events into a list at once? What would happen if we loaded $10^7$ events into RAM?
> 2. **Numerical Trigonometry Question**: Why do we use `np.arctan2(py, px)` instead of `np.arctan(py / px)` when calculating azimuthal angle $\phi$?


## Step 2: Extract Partons & Compute Kinematic Observables

We iterate over events, extract final-state $b$ and $\bar{b}$ partons ($PID = \pm 5, status = 1$), reconstruct $p_Z^\mu = p_b^\mu + p_{\bar{b}}^\mu$, and compute kinematic observables ($p_T^Z, p_T^b, \eta_b, \phi_b, \Delta R_{b\bar{b}}, m_{b\bar{b}}$).

In [ ]:
def wrap_phi(phi):
    return np.arctan2(np.sin(phi), np.cos(phi))

z_pt_list = []
b_pt_list = []
bbar_pt_list = []
b_eta_list = []
bbar_eta_list = []
delta_r_list = []
m_bb_list = []
weights_list = []

print(f"Processing LHE events from {target_lhe}...")
events = pylhe.read_lhe_with_attributes(target_lhe)

for event in events:
    weight = getattr(event.eventinfo, 'weight', 1.0)
    b_part, bbar_part = None, None

    for p in event.particles:
        if p.status == 1 and p.id == 5:
            b_part = p
        elif p.status == 1 and p.id == -5:
            bbar_part = p

    if b_part is not None and bbar_part is not None:
        px_b, py_b, pz_b, e_b = b_part.px, b_part.py, b_part.pz, b_part.e
        px_bbar, py_bbar, pz_bbar, e_bbar = bbar_part.px, bbar_part.py, bbar_part.pz, bbar_part.e

        # Reconstruct Z = b + bbar
        px_z = px_b + px_bbar
        py_z = py_b + py_bbar
        pz_z = pz_b + pz_bbar
        e_z = e_b + e_bbar

        pt_z = np.hypot(px_z, py_z)
        m_z = np.sqrt(max(0.0, e_z**2 - (px_z**2 + py_z**2 + pz_z**2)))

        pt_b = np.hypot(px_b, py_b)
        p_b = np.sqrt(px_b**2 + py_b**2 + pz_b**2)
        eta_b = 0.5 * np.log((p_b + pz_b) / max(1e-9, p_b - pz_b))
        phi_b = np.arctan2(py_b, px_b)

        pt_bbar = np.hypot(px_bbar, py_bbar)
        p_bbar = np.sqrt(px_bbar**2 + py_bbar**2 + pz_bbar**2)
        eta_bbar = 0.5 * np.log((p_bbar + pz_bbar) / max(1e-9, p_bbar - pz_bbar))
        phi_bbar = np.arctan2(py_bbar, px_bbar)

        d_eta = eta_b - eta_bbar
        d_phi = wrap_phi(phi_b - phi_bbar)
        delta_r = np.hypot(d_eta, d_phi)

        z_pt_list.append(pt_z)
        b_pt_list.append(pt_b)
        bbar_pt_list.append(pt_bbar)
        b_eta_list.append(eta_b)
        bbar_eta_list.append(eta_bbar)
        delta_r_list.append(delta_r)
        m_bb_list.append(m_z)
        weights_list.append(weight)

z_pt = np.array(z_pt_list)
b_pt = np.array(b_pt_list)
bbar_pt = np.array(bbar_pt_list)
b_eta = np.array(b_eta_list)
bbar_eta = np.array(bbar_eta_list)
delta_r = np.array(delta_r_list)
m_bb = np.array(m_bb_list)
weights = np.array(weights_list)

print(f"Processed {len(z_pt)} events.")
print(f"Mean Z pT: {np.average(z_pt, weights=weights):.2f} GeV")
print(f"Mean m(bb): {np.average(m_bb, weights=weights):.2f} GeV")
print(f"Mean Delta R(b, bbar): {np.average(delta_r, weights=weights):.2f}")


> [!IMPORTANT]
> ### Self-Reflection & Physics Checkpoint 2.2
> 1. **Physics Question**: Why is the reconstructed invariant mass $m_{b\bar{b}}$ peaked at $91.2\text{ GeV}$? What physical effect causes events to scatter away from the exact peak?
> 2. **Coding Question**: How does `plt.hist(..., weights=weights)` handle Monte Carlo event weighting? Why would ignoring `weights` produce incorrect physics histograms for weighted MC samples?


## Step 3: Plot 1D Kinematic Histograms with `mplhep`

We plot 1D distributions for $p_T^Z$, $p_T(b)$, $\eta$, $m_{b\bar{b}}$, and $\Delta R_{b\bar{b}}$, applying Monte Carlo weights.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Generator-Level Kinematic Observables: $pp \\to Z+j, Z \\to b\\bar{b}$", fontsize=16)

# 1. Z pT
axs[0, 0].hist(z_pt, bins=30, range=(100, 600), weights=weights, histtype='step', linewidth=2, color='navy')
axs[0, 0].set_xlabel("Reconstructed $p_T^Z$ [GeV]")
axs[0, 0].set_ylabel("Weighted Events")
axs[0, 0].axvline(150, color='red', linestyle='--', label='$p_T^Z > 150$ GeV Cut')
axs[0, 0].legend()

# 2. b-quark pT
axs[0, 1].hist(b_pt, bins=30, range=(0, 400), weights=weights, histtype='step', linewidth=2, color='crimson', label='$b$ quark')
axs[0, 1].hist(bbar_pt, bins=30, range=(0, 400), weights=weights, histtype='step', linewidth=2, color='darkorange', linestyle=':', label='$\\bar{b}$ quark')
axs[0, 1].set_xlabel("$p_T(b)$ [GeV]")
axs[0, 1].set_ylabel("Weighted Events")
axs[0, 1].legend()

# 3. b-quark pseudorapidity
axs[0, 2].hist(b_eta, bins=30, range=(-4, 4), weights=weights, histtype='step', linewidth=2, color='purple', label='$\\eta(b)$')
axs[0, 2].hist(bbar_eta, bins=30, range=(-4, 4), weights=weights, histtype='step', linewidth=2, color='teal', linestyle=':', label='$\\eta(\\bar{b})$')
axs[0, 2].set_xlabel("Pseudorapidity $\\eta$")
axs[0, 2].set_ylabel("Weighted Events")
axs[0, 2].legend()

# 4. Invariant Mass m(bb)
axs[1, 0].hist(m_bb, bins=30, range=(60, 120), weights=weights, histtype='step', linewidth=2, color='darkgreen')
axs[1, 0].set_xlabel("Invariant Mass $m_{b\\bar{b}}$ [GeV]")
axs[1, 0].set_ylabel("Weighted Events")
axs[1, 0].axvline(91.1876, color='black', linestyle='--', label='PDG $m_Z = 91.2$ GeV')
axs[1, 0].legend()

# 5. Delta R(b, bbar)
axs[1, 1].hist(delta_r, bins=30, range=(0, 3.5), weights=weights, histtype='step', linewidth=2, color='chocolate')
axs[1, 1].set_xlabel("Angular Separation $\\Delta R_{b\\bar{b}}$")
axs[1, 1].set_ylabel("Weighted Events")

# 6. Analytical Guide Info Panel
axs[1, 2].axis('off')
guide_info = (
    "High-Boost Opening Angle Guide:\n"
    "$\\Delta R_{b\\bar{b}} \\approx \\frac{2 m_Z}{p_T^Z}$\n\n"
    f"At 250 GeV: $\\Delta R \\approx {2*91.1876/250:.2f}$\n"
    f"At 400 GeV: $\\Delta R \\approx {2*91.1876/400:.2f}$\n"
    f"At 600 GeV: $\\Delta R \\approx {2*91.1876/600:.2f}$\n\n"
    "Notice how high pT Z bosons produce\n"
    "collimated b-quark pairs!"
)
axs[1, 2].text(0.1, 0.25, guide_info, fontsize=12, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()


## Step 4: 2D Correlation $\Delta R_{b\bar{b}}$ vs. $p_T^Z$ & Boost Guide Overlay

As $p_T^Z$ increases, relativistic boost collimates the decay products ($b, \bar{b}$). We overlay the analytical high-boost two-body guide line:
$$\Delta R_{b\bar{b}} = \frac{2m_Z}{p_T^Z}$$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

h = ax.hist2d(z_pt, delta_r, bins=[30, 30], range=[[100, 600], [0, 3.5]], weights=weights, cmap='viridis')
plt.colorbar(h[3], ax=ax, label='Weighted Events')

pt_grid = np.linspace(120, 600, 200)
m_z_pdg = 91.1876
delta_r_guide = (2.0 * m_z_pdg) / pt_grid

ax.plot(pt_grid, delta_r_guide, color='red', linestyle='--', linewidth=2.5, label='High-Boost Guide: $\\Delta R = \\frac{2 m_Z}{p_T^Z}$')

ax.set_xlabel("Reconstructed $p_T^Z$ [GeV]", fontsize=13)
ax.set_ylabel("Angular Separation $\\Delta R_{b\\bar{b}}$", fontsize=13)
ax.set_title("2D Correlation: $\\Delta R_{b\\bar{b}}$ vs. $p_T^Z$ in $pp \\to Z+j, Z \\to b\\bar{b}$", fontsize=14)
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


> [!IMPORTANT]
> ### Section Checkpoint & Quantitative Exercise 2.3
> 1. **Collimation Calculation**: Compute the approximate opening angle $\Delta R \approx \frac{2m_Z}{p_T^Z}$ for $p_T^Z = 250\text{ GeV}$, $400\text{ GeV}$, and $600\text{ GeV}$.
>    - *250 GeV*: $\Delta R \approx 0.73$
>    - *400 GeV*: $\Delta R \approx 0.46$
>    - *600 GeV*: $\Delta R \approx 0.30$
> 2. **Large-$R$ Jet Threshold**: Standard small-$R$ jets use $R=0.4$, while large-$R$ jets use $R=0.8$ or $R=1.0$. At what $p_T^Z$ threshold will both $b$ quarks begin to fall inside a single $R=0.8$ large-$R$ jet?
>    - *Answer*: Since $\Delta R \lesssim 0.8$, we require $\frac{2m_Z}{p_T^Z} \lesssim 0.8 \implies p_T^Z \gtrsim \frac{2 \times 91.2}{0.8} \approx 228\text{ GeV}$. For $p_T^Z > 200\text{ GeV}$, both decay partons merge into a single large-$R$ jet!
